# Hallow 2017 GFR GSA model, 86D

This notebook exposes the current 86-dimensional single-code-parameter input vector for GSA with `GFR_ml_min` as the primary output. Feedback mechanisms remain in the model, while non-sampled feedback and scenario parameters stay fixed at baseline values.

In [ ]:
# GSA input block: edit or overwrite these 86 values for one model run.
GSA_OUTPUT = 'GFR_ml_min'
GSA_INPUT_BASELINE = {
    'BV_filling_pressure_slope': 7.436,
    'CD_flow_dependence': 0.85,
    'C_Na_error': 0.16666666666666666,
    'C_P_bowmans': 1000.0,
    'C_P_oncotic': 1000.0,
    'C_cardiac_output_delayed': 0.001,
    'C_co_error': 1e-05,
    'C_prerenal_blood_pressure': 1000.0,
    'C_pt_water': 1000.0,
    'C_rbf': 1000.0,
    'C_rihp': 0.1,
    'C_tgf': 1.0,
    'DCT_flow_dependence': 0.85,
    'Dc_cd': 1.7e-05,
    'Dc_dt': 1.7e-05,
    'Dc_lh': 1.7e-05,
    'Dc_pt_nom': 2.7e-05,
    'F_md_scale_tubulo_glomerular_feedback': 6.0,
    'Ki_CO': 30.0,
    'Ki_VP': 0.01,
    'Kp_CO': 1.5,
    'Kp_VP': 0.1,
    'L_cd': 0.003,
    'L_dct': 0.004,
    'L_lh_asc': 0.003,
    'L_lh_des': 0.003,
    'L_pt_s1_nom': 0.002,
    'L_pt_s2_nom': 0.002,
    'L_pt_s3_nom': 0.002,
    'LoH_flow_dependence': 0.85,
    'Na_controller_gain': 2.0,
    'Na_intake_rate': 0.0014,
    'P_venous': 4.0,
    'Pc_cd_mmHg': 5.0,
    'Pc_dt_mmHg': 6.0,
    'Pc_lh_asc_mmHg': 7.0,
    'Pc_lh_des_mmHg': 8.0,
    'Pc_pt_s1_mmHg': 14.0,
    'Pc_pt_s2_mmHg': 12.0,
    'Pc_pt_s3_mmHg': 11.0,
    'RBF_autoreg_scale': 0.0,
    'RBF_autoreg_steepness': 0.001,
    'R_venous': 400.0,
    'S_tubulo_glomerular_feedback': 0.6,
    'afferent_diameter_range': 1.2e-05,
    'afferent_signal_nonlin_scale': 3.0,
    'anp_cd_slope': 0.055,
    'baseline_nephrons': 50000.0,
    'efferent_diameter_range': 3e-06,
    'efferent_signal_nonlin_scale': 3.0,
    'gamma': 1.16667e-05,
    'glucose_concentration': 5.5,
    'interstitial_oncotic_pressure': 5.0,
    'map_rsna_slope': 5.0,
    'myogenic_steepness': 2.0,
    'nom_ADH_urea_permeability': 0.94,
    'nom_ADH_water_permeability': 0.94,
    'nom_Kf': 3.9,
    'nom_afferent_diameter': 1.5e-05,
    'nom_efferent_diameter': 9.25e-06,
    'nom_glucose_reabs_per_unit_length_s1': 0.0002,
    'nom_glucose_reabs_per_unit_length_s2': 0.0,
    'nom_glucose_reabs_per_unit_length_s3': 2.5e-05,
    'nom_preafferent_arteriole_resistance': 3500.0,
    'nom_renal_blood_flow_L_min': 0.011,
    'nom_right_atrial_pressure': 0.87,
    'nom_water_intake': 2.0,
    'nominal_dt_na_reabsorption': 0.5,
    'nominal_loh_na_reabsorption': 0.88,
    'nominal_pt_na_reabsorption': 0.7,
    'plasma_protein_concentration': 6.2,
    'plasma_urea': 5.0,
    'preaff_diameter_range': 0.25,
    'preaff_signal_nonlin_scale': 3.0,
    'rap_anp_slope': 1.0,
    'rap_rsna_slope': 0.008,
    'reference_BV_mean_filling_pressure': 30.18,
    'rsna_preaff_EC50': 1.0,
    'rsna_preaff_hill': 4.0,
    'rsna_preaff_vmax': 0.0,
    'scale_aldo_CD': 1.0,
    'scale_aldo_DT': 1.0,
    'tissue_autoreg_scale': 1.0,
    'tubular_compliance': 0.2,
    'urea_permeability_PT': 0.5,
    'viscosity_length_constant': 1.5e-09,
}

GSA_INPUT_NAMES = list(GSA_INPUT_BASELINE)
GSA_INPUT_DIMENSION = len(GSA_INPUT_NAMES)
assert GSA_INPUT_DIMENSION == 86
GSA_INPUT_VECTOR_BASELINE = [GSA_INPUT_BASELINE[name] for name in GSA_INPUT_NAMES]
print(f'GSA input dimension: {GSA_INPUT_DIMENSION}')
print(f'Primary output: {GSA_OUTPUT}')


In [ ]:
from pathlib import Path
import sys

LOCAL_DEPS = Path.cwd().resolve() / 'local_python_deps'
if LOCAL_DEPS.exists() and str(LOCAL_DEPS) not in sys.path:
    sys.path.insert(0, str(LOCAL_DEPS))
import math
import json
import warnings

import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp

ROOT = Path.cwd().resolve()
FIGURE_DIR = ROOT / "figure"
OUT_DIR = ROOT / "out"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
R_DATA_DIR = Path(r'C:\Users\ASUS\Documents\New project\Hallow_2017\Data_csv')

print(f"Notebook root: {ROOT}")
print(f"Figures:      {FIGURE_DIR}")
print(f"CSV outputs:  {OUT_DIR}")


In [ ]:
PARAMS = {'ACE_chymase_fraction': 0.95, 'AT1_PRC_slope': -1.2, 'AT1_PRC_yint': 0.0, 'AT1_PT_slope': 0.002, 'AT1_aff_slope': 0.005, 'AT1_aldo_slope': 0.02, 'AT1_bound_AngII_half_life': 0.2, 'AT1_eff_slope': 0.01, 'AT1_preaff_slope': 0.0, 'AT1_svr_slope': 0.0, 'AT2_bound_AngII_half_life': 0.2, 'AngII_half_life': 0.011, 'AngI_half_life': 0.008333, 'AscLoH_Reab_Rate': 0.0012282492444444443, 'BV_filling_pressure_slope': 7.436, 'BV_scale_species': 0.0036, 'CA_inhibitor': 1.0, 'CCB_effect_on_afferent_resistance': 1.0, 'CCB_effect_on_efferent_resistance': 1.0, 'CCB_effect_on_preafferent_resistance': 1.0, 'CD_PN_loss_rate': 0.0, 'CD_flow_dependence': 0.85, 'CO_nom': 0.04, 'CO_scale_species': 0.008, 'C_Na_error': 0.16666666666666666, 'C_P_bowmans': 1000.0, 'C_P_oncotic': 1000.0, 'C_aldo_secretion': 1000.0, 'C_cardiac_output_delayed': 0.001, 'C_co_error': 1e-05, 'C_na_excretion_na_amount': -0.03333333333333333, 'C_na_intake_na_amount': 0.03333333333333333, 'C_prerenal_blood_pressure': 1000.0, 'C_pt_water': 1000.0, 'C_rbf': 1000.0, 'C_rihp': 0.1, 'C_serum_creatinine': 1.0, 'C_tgf': 1.0, 'C_tgf_reset': 0.0, 'C_urine_flow_ecf_volume': -0.03333333333333333, 'C_water_intake_ecf_volume': 0.03333333333333333, 'DCT_flow_dependence': 0.85, 'DRI_effect_on_PRA': 1.0, 'Dc_cd': 1.7e-05, 'Dc_dt': 1.7e-05, 'Dc_lh': 1.7e-05, 'Dc_pt_nom': 2.7e-05, 'ECF_nom': 0.06, 'ECF_scale_species': 0.004, 'F_md_scale_tubulo_glomerular_feedback': 6.0, 'HCTZ_effect_on_DT_Na_reabs': 1.0, 'HCTZ_effect_on_renin_secretion': 1.0, 'Ki_CO': 30.0, 'Ki_VP': 0.01, 'Kp_CO': 1.5, 'Kp_VP': 0.1, 'L_cd': 0.003, 'L_dL': 10.0, 'L_dct': 0.004, 'L_lh_asc': 0.003, 'L_lh_des': 0.003, 'L_m3': 0.001, 'L_mL': 1000.0, 'L_pt_s1_nom': 0.002, 'L_pt_s2_nom': 0.002, 'L_pt_s3_nom': 0.002, 'LoH_flow_dependence': 0.85, 'MD_Na_concentration_setpoint': 60.0, 'MR_antagonist_effect_on_aldo_MR': 1.0, 'MW_creatinine': 113.12, 'Na_controller_gain': 2.0, 'Na_intake_rate': 0.0014, 'PT_Na_reab_perUnitSA_0': 9.603448577145201, 'P_renal_vein': 4.0, 'P_venous': 4.0, 'Pc_cd_mmHg': 5.0, 'Pc_dt_mmHg': 6.0, 'Pc_lh_asc_mmHg': 7.0, 'Pc_lh_des_mmHg': 8.0, 'Pc_pt_s1_mmHg': 14.0, 'Pc_pt_s2_mmHg': 12.0, 'Pc_pt_s3_mmHg': 11.0, 'Pi': 3.1416, 'RBF_autoreg_scale': 0.0, 'RBF_autoreg_steepness': 0.001, 'RIHP0': 6.0, 'R_venous': 400.0, 'SGLT1_inhibition': 1.0, 'SGLT2_inhibition': 1.0, 'S_tubulo_glomerular_feedback': 0.6, 'T_PT_Na_reabs_PT_diameter': 10000000000.0, 'T_PT_Na_reabs_PT_length': 10000000000.0, 'T_glomerular_pressure_increases_Kf': 10000000000.0, 'afferent_diameter_range': 1.2e-05, 'afferent_signal_nonlin_scale': 3.0, 'anp_aff_resistance_slope': 0.4125, 'anp_cd_slope': 0.055, 'anp_eff_resistance_slope': 0.0825, 'anp_preaff_resistance_slope': 0.4125, 'baseline_nephrons': 50000.0, 'concentration_to_renin_activity_conversion_plasma': 61.0, 'diabetic_adaptation': 1.0, 'dl_ml': 0.01, 'efferent_diameter_range': 3e-06, 'efferent_signal_nonlin_scale': 3.0, 'equilibrium_serum_creatinine': 0.8, 'fraction_AT1_bound_AngII': 0.75, 'g_mg': 0.001, 'gamma': 1.16667e-05, 'glucose_concentration': 5.5, 'gp_autoreg_scale': 2.0, 'hill_aldo_CD': 0.02, 'hill_aldo_DT': 0.1, 'hr_day': 24.0, 'interstitial_oncotic_pressure': 5.0, 'map_rsna_slope': 5.0, 'max_albumin_reabsorption_fraction': 0.98448, 'maximal_glom_surface_area_increase': 0.0, 'maximal_tubule_diameter_increase': 0.0, 'maximal_tubule_length_increase': 0.0, 'maximum_reabsorption_capacity': 5.5e-07, 'md_renin_A': 1.0, 'md_renin_tau': 1.25, 'min_day': 1440.0, 'min_hr': 60.0, 'myogenic_steepness': 2.0, 'nL_mL': 1000000.0, 'ng_mg': 1e-06, 'nom_ADH_urea_permeability': 0.94, 'nom_ADH_water_permeability': 0.94, 'nom_ANP': 1.0, 'nom_DT_Na_outflow': 0.0062808199999999995, 'nom_GFR': 2.4923888888888888, 'nom_Kf': 3.9, 'nom_LoH_Na_outflow': 0.012561639999999999, 'nom_Na_in_AscLoH': 2.0936066666666666e-06, 'nom_Na_in_CD': 1.256164e-07, 'nom_Na_in_DCT': 2.512328e-07, 'nom_PT_Na_outflow': 0.10468033333333333, 'nom_RVR': 8727.272727272728, 'nom_TPR': 2500.0, 'nom_afferent_diameter': 1.5e-05, 'nom_efferent_diameter': 9.25e-06, 'nom_filtered_sodium_load': 0.3489344444444444, 'nom_glomerular_albumin_sieving_coefficient': 0.0001, 'nom_glomerular_pressure': 54.98148148148148, 'nom_glucose_reabs_per_unit_length_s1': 0.0002, 'nom_glucose_reabs_per_unit_length_s2': 0.0, 'nom_glucose_reabs_per_unit_length_s3': 2.5e-05, 'nom_oncotic_pressure_difference': 28.2, 'nom_oncotic_pressure_peritubular': 28.2, 'nom_peritubular_cap_Kf': 0.0001282750827867435, 'nom_peritubular_resistance': 536.8464907189164, 'nom_postglomerular_pressure': 9.90531139790808, 'nom_preafferent_arteriole_resistance': 3500.0, 'nom_preafferent_pressure': 61.5, 'nom_renal_blood_flow_L_min': 0.011, 'nom_right_atrial_pressure': 0.87, 'nom_rsna': 1.0, 'nom_systemic_arterial_resistance': 2100.0, 'nom_water_intake': 2.0, 'nominal_ACE_activity': 48.9, 'nominal_AT1_bound_AngII_degradation_rate': 3.465735902799726, 'nominal_AT1_bound_AngII_half_life': 0.2, 'nominal_AT1_receptor_binding_rate': 12.1, 'nominal_AT2_bound_AngII_degradation_rate': 3.465735902799726, 'nominal_AT2_bound_AngII_half_life': 0.2, 'nominal_AT2_receptor_binding_rate': 4.0, 'nominal_AngII_degradation_rate': 63.01338005090411, 'nominal_AngII_half_life': 0.011000000000000001, 'nominal_AngI_degradation_rate': 83.17766166719343, 'nominal_AngI_half_life': 0.008333333333333333, 'nominal_aldosterone_concentration': 85.0, 'nominal_cd_na_reabsorption': 0.7770991685798988, 'nominal_chymase_activity': 1.25, 'nominal_dt_na_reabsorption': 0.5, 'nominal_equilibrium_AT1_bound_AngII': 16.6315288606173, 'nominal_equilibrium_AT2_bound_AngII': 5.543842953539081, 'nominal_equilibrium_AngI': 7.5, 'nominal_equilibrium_AngII': 4.75, 'nominal_equilibrium_PRA': 1000.0, 'nominal_equilibrium_PRC': 16.4, 'nominal_loh_na_reabsorption': 0.88, 'nominal_map_setpoint': 100.0, 'nominal_pt_na_reabsorption': 0.7, 'nominal_renin_half_life': 0.1733, 'nominal_vasopressin_conc': 4.0, 'plasma_albumin_concentration': 3.5, 'plasma_protein_concentration': 6.2, 'plasma_urea': 5.0, 'potassium_concentration': 5.0, 'preaff_autoreg_scale': 2.0, 'preaff_diameter_range': 0.25, 'preaff_signal_nonlin_scale': 3.0, 'pressure_natriuresis_CD_scale': 2.0, 'pressure_natriuresis_CD_slope': 1.0, 'pressure_natriuresis_DCT_scale': 0.0, 'pressure_natriuresis_DCT_slope': 1.0, 'pressure_natriuresis_LoH_scale': 1.0, 'pressure_natriuresis_LoH_slope': 1.0, 'pressure_natriuresis_PT_scale': 0.2, 'pressure_natriuresis_PT_slope': 1.0, 'pt_rsna_scale': 1.0, 'pt_rsna_slope': 1.0, 'rap_anp_slope': 1.0, 'rap_rsna_slope': 0.008, 'ref_Na_concentration': 140.0, 'reference_BV_mean_filling_pressure': 30.18, 'renal_threshold_Na_reabs': 1.6e-05, 'renin_half_life': 0.1733, 'rsna_preaff_EC50': 1.0, 'rsna_preaff_hill': 4.0, 'rsna_preaff_vmax': 0.0, 'rsna_renin_slope': 0.36, 'scale_aldo_CD': 1.0, 'scale_aldo_DT': 1.0, 'secs_mins': 60.0, 'tissue_autoreg_scale': 1.0, 'tubular_compliance': 0.2, 'tubular_reabsorption': 0.0024750277777777774, 'urea_permeability_PT': 0.5, 'viscosity_length_constant': 1.5e-09, 'water_intake_species_scale': 0.0125, 'water_intake_vasopressin_scale': 1.5, 'water_intake_vasopressin_slope': -0.5, 'rsna_effect_on_aff_resistance': 1.0, 'PT_Na_reabs_effect_increasing_tubular_length': 0.0, 'PT_Na_reabs_effect_increasing_tubular_diameter': 0.0, 'mmHg_Nperm2_conv': 133.32, 'pi': 3.14}

STATES = ['AngI', 'AngII', 'AT1_bound_AngII', 'AT2_bound_AngII', 'plasma_renin_concentration', 'pre_renal_blood_pressure', 'extracellular_fluid_volume', 'sodium_amount', 'tubulo_glomerular_feedback_effect', 'normalized_aldosterone_level', 'preafferent_pressure_autoreg_signal', 'glomerular_pressure_autoreg_signal', 'F_out_dt_delay', 'cardiac_output_delayed', 'CO_error', 'Na_concentration_error', 'F0_TGF', 'P_bowmans', 'oncotic_pressure_difference', 'renal_blood_flow_L_min_delayed', 'renal_interstitial_hydrostatic_pressure', 'disease_effects_increasing_Kf', 'disease_effects_decreasing_CD_PN', 'tubular_length_increase', 'tubular_diameter_increase', 'water_out_s1_delayed', 'water_out_s2_delayed', 'water_out_s3_delayed', 'reabsorbed_urea_cd_delayed', 'UGE', 'cumWaterExcretion']
IDX = {name: i for i, name in enumerate(STATES)}

FIGURE2_OVERRIDES = {
    'pt_rsna_scale': 0.0, 'anp_aff_resistance_slope': 0.0,
    'anp_eff_resistance_slope': 0.0, 'anp_preaff_resistance_slope': 0.0,
    'gp_autoreg_scale': 0.0, 'preaff_autoreg_scale': 0.0,
    'AT1_preaff_slope': 0.0, 'AT1_aff_slope': 0.0, 'AT1_PT_slope': 0.0,
    'AT1_aldo_slope': 0.0, 'AT1_eff_slope': 0.0,
    'hill_aldo_DT': 0.0, 'hill_aldo_CD': 0.0,
    'pressure_natriuresis_CD_scale': 0.0, 'pressure_natriuresis_DCT_scale': 0.0,
    'pressure_natriuresis_PT_scale': 0.0, 'pressure_natriuresis_LoH_scale': 0.0,
    'md_renin_tau': 0.0, 'rsna_renin_slope': 0.0,
    'nom_ADH_water_permeability': 0.99, 'nom_ADH_urea_permeability': 0.98,
    'Na_controller_gain': 0.1, 'Ki_VP': 0.0001, 'Kp_VP': 1.0,
    'water_intake_vasopressin_scale': 0.0, 'LoH_flow_dependence': 0.9,
    'MD_Na_concentration_setpoint': 39.0,
}

def apply_figure2_configuration(base, tgf_strength=0.7, ca_inhibitor=1.0):
    p = dict(base)
    p.update(FIGURE2_OVERRIDES)
    p['S_tubulo_glomerular_feedback'] = float(tgf_strength)
    p['CA_inhibitor'] = float(ca_inhibitor)
    return p

print(f'{len(PARAMS)} parameters; {len(STATES)} dynamic states')


In [ ]:
def safe_exp(x):
    return math.exp(max(-700.0, min(700.0, float(x))))

def safe_log(x):
    return math.log(max(float(x), np.finfo(float).tiny))

def safe_log10(x):
    return math.log10(max(float(x), np.finfo(float).tiny))

def max_fn(*values):
    return max(float(v) for v in values)

def min_fn(*values):
    return min(float(v) for v in values)

SAFE_GLOBALS = {
    '__builtins__': {}, 'safe_exp': safe_exp, 'safe_log': safe_log,
    'safe_log10': safe_log10, 'max_fn': max_fn, 'min_fn': min_fn,
}

def run_kernel(context, compiled_kernel):
    exec(compiled_kernel, SAFE_GLOBALS, context)


In [ ]:
SYSTEMIC_SOURCE = r'''
number_of_functional_nephrons = baseline_nephrons
tissue_autoregulation_signal = max_fn(0.1,1+tissue_autoreg_scale*((Kp_CO/CO_scale_species)*(cardiac_output_delayed - CO_nom)+(Ki_CO/CO_scale_species)*CO_error))
AT1_svr_int = 1 - AT1_svr_slope*nominal_equilibrium_AT1_bound_AngII
AT1_bound_AngII_effect_on_SVR = AT1_svr_int + AT1_svr_slope * AT1_bound_AngII
systemic_arterial_resistance = tissue_autoregulation_signal*(nom_systemic_arterial_resistance)*AT1_bound_AngII_effect_on_SVR
resistance_to_venous_return = ((8 * R_venous + systemic_arterial_resistance) / 31)
blood_volume_L = BV_scale_species*(4.56+2.43/(1+safe_exp(-((extracellular_fluid_volume/ECF_scale_species-18.1))*0.474)))
mean_filling_pressure = (((BV_filling_pressure_slope/BV_scale_species) * blood_volume_L - reference_BV_mean_filling_pressure))
venous_return = ((mean_filling_pressure) / resistance_to_venous_return)
cardiac_output = venous_return
right_atrial_pressure = (nom_right_atrial_pressure * safe_exp(blood_volume_L - 5*BV_scale_species))
aldosterone_concentration = normalized_aldosterone_level* nominal_aldosterone_concentration
Aldo_MR_normalised_effect = normalized_aldosterone_level*MR_antagonist_effect_on_aldo_MR
MAP_effect_on_rsna = 0.5 + 1.0 / (1 + safe_exp((pre_renal_blood_pressure - nominal_map_setpoint) / map_rsna_slope))
R_atrial_pressure_effect_on_rsna_int = 1+rap_rsna_slope*nom_right_atrial_pressure
R_atrial_pressure_effect_on_rsna = R_atrial_pressure_effect_on_rsna_int - rap_rsna_slope * right_atrial_pressure
renal_sympathetic_nerve_activity = nom_rsna*MAP_effect_on_rsna * R_atrial_pressure_effect_on_rsna
rap_anp_intercept = 1 + rap_anp_slope/2
normalized_atrial_NP_concentration = nom_ANP*((rap_anp_intercept - rap_anp_slope / (1 + safe_exp(right_atrial_pressure - nom_right_atrial_pressure))))
AT1_preaff_int = 1-AT1_preaff_slope*nominal_equilibrium_AT1_bound_AngII
At1_effect_on_preaff_resistance = AT1_preaff_int  + AT1_preaff_slope *  AT1_bound_AngII
ANP_effect_on_preaff = 1 - anp_preaff_resistance_slope * (normalized_atrial_NP_concentration - nom_ANP)
rsna_preaff_intercept = 1-rsna_preaff_vmax/2
rsna_effect_on_preaff_resistance = rsna_preaff_intercept + rsna_preaff_vmax*(renal_sympathetic_nerve_activity**rsna_preaff_hill)/(rsna_preaff_EC50**rsna_preaff_hill + renal_sympathetic_nerve_activity**rsna_preaff_hill)
preaff_arteriole_signal_multiplier = At1_effect_on_preaff_resistance*rsna_effect_on_preaff_resistance*ANP_effect_on_preaff*(preafferent_pressure_autoreg_signal)*CCB_effect_on_preafferent_resistance
preaff_arteriole_adjusted_signal_multiplier = (1/(1+safe_exp(preaff_signal_nonlin_scale*(preaff_arteriole_signal_multiplier-1)))+0.5)
nom_effective_preaff_diameter = nom_preafferent_arteriole_resistance**(-0.25)
preaff_diameter = nom_effective_preaff_diameter+preaff_diameter_range*(preaff_arteriole_adjusted_signal_multiplier-1)
preafferent_arteriole_resistance = 1/(preaff_diameter **4)
AT1_aff_int = 1 - AT1_aff_slope*nominal_equilibrium_AT1_bound_AngII
AT1_effect_on_afferent_resistance = AT1_aff_int + AT1_aff_slope * AT1_bound_AngII
rsna_effect_on_aff_resistance = 1
ANP_effect_on_afferent = 1 - anp_aff_resistance_slope * (normalized_atrial_NP_concentration - nom_ANP)
afferent_arteriole_signal_multiplier = rsna_effect_on_aff_resistance * tubulo_glomerular_feedback_effect * AT1_effect_on_afferent_resistance * ANP_effect_on_afferent*glomerular_pressure_autoreg_signal*CCB_effect_on_afferent_resistance
afferent_arteriole_adjusted_signal_multiplier = (1/(1+safe_exp(afferent_signal_nonlin_scale*(afferent_arteriole_signal_multiplier-1)))+0.5)
afferent_diameter = nom_afferent_diameter+afferent_diameter_range*(afferent_arteriole_adjusted_signal_multiplier-1)
afferent_arteriole_resistance = L_m3*viscosity_length_constant/(afferent_diameter**4)
AT1_eff_int = 1 - AT1_eff_slope*nominal_equilibrium_AT1_bound_AngII
AT1_bound_AngII_effect_on_efferent = AT1_eff_int + AT1_eff_slope * AT1_bound_AngII
ANP_effect_on_efferent_resistance = 1 + anp_eff_resistance_slope * (normalized_atrial_NP_concentration - nom_ANP)
efferent_arteriole_signal_multiplier = (((AT1_bound_AngII_effect_on_efferent * ANP_effect_on_efferent_resistance ))*CCB_effect_on_efferent_resistance  )
efferent_arteriole_adjusted_signal_multiplier = 1/(1+safe_exp(efferent_signal_nonlin_scale*(efferent_arteriole_signal_multiplier-1)))+0.5
efferent_diameter = nom_efferent_diameter+efferent_diameter_range*(efferent_arteriole_adjusted_signal_multiplier-1)
efferent_arteriole_resistance = L_m3*viscosity_length_constant/(efferent_diameter**4)
RBF_autoreg_int = 1 - RBF_autoreg_scale/2
peritubular_autoreg_signal = RBF_autoreg_int + RBF_autoreg_scale/(1+safe_exp((nom_renal_blood_flow_L_min - renal_blood_flow_L_min_delayed)/RBF_autoreg_steepness))
autoregulated_peritubular_resistance = peritubular_autoreg_signal*nom_peritubular_resistance
renal_vascular_resistance = (preafferent_arteriole_resistance + (afferent_arteriole_resistance + efferent_arteriole_resistance) / number_of_functional_nephrons + autoregulated_peritubular_resistance)
total_peripheral_resistance_old = (systemic_arterial_resistance * renal_vascular_resistance / (systemic_arterial_resistance + renal_vascular_resistance) + R_venous)
total_peripheral_resistance = systemic_arterial_resistance + R_venous
mean_arterial_pressure_MAP = (cardiac_output * total_peripheral_resistance)
renal_blood_flow_L_min = ((mean_arterial_pressure_MAP - P_venous) / renal_vascular_resistance)
renal_blood_flow_ml_hr = renal_blood_flow_L_min * 1000 * 60
preafferent_pressure = mean_arterial_pressure_MAP - renal_blood_flow_L_min*preafferent_arteriole_resistance
glomerular_pressure = (mean_arterial_pressure_MAP - renal_blood_flow_L_min * (preafferent_arteriole_resistance + afferent_arteriole_resistance / number_of_functional_nephrons))
postglomerular_pressure = (mean_arterial_pressure_MAP - renal_blood_flow_L_min * (preafferent_arteriole_resistance + (afferent_arteriole_resistance+efferent_arteriole_resistance) / number_of_functional_nephrons))
GP_effect_increasing_Kf = (maximal_glom_surface_area_increase - disease_effects_increasing_Kf) * max_fn(glomerular_pressure/(nom_glomerular_pressure+2) - 1,0) / T_glomerular_pressure_increases_Kf
glomerular_hydrostatic_conductance_Kf = nom_Kf*(1+disease_effects_increasing_Kf)
gp_autoreg_int = 1 - gp_autoreg_scale/2
preaff_autoreg_int = 1 - preaff_autoreg_scale/2
preafferent_pressure_autoreg_function = preaff_autoreg_int+preaff_autoreg_scale/(1+safe_exp((nom_preafferent_pressure - preafferent_pressure)/myogenic_steepness))
glomerular_pressure_autoreg_function = gp_autoreg_int+gp_autoreg_scale/(1+safe_exp((nom_glomerular_pressure - glomerular_pressure)/myogenic_steepness))
net_filtration_pressure = glomerular_pressure - oncotic_pressure_difference - P_bowmans
SNGFR_nL_min = glomerular_hydrostatic_conductance_Kf * (glomerular_pressure - oncotic_pressure_difference - P_bowmans)
GFR = (SNGFR_nL_min / 1000 / 1000000 * number_of_functional_nephrons)
GFR_ml_min = GFR * 1000
Oncotic_pressure_in = 1.629*plasma_protein_concentration+0.2935*(plasma_protein_concentration**2)
SNRBF_nl_min = 1e6*1000*renal_blood_flow_L_min/number_of_functional_nephrons
plasma_protein_concentration_out = SNRBF_nl_min*plasma_protein_concentration/(SNRBF_nl_min-SNGFR_nL_min)
Oncotic_pressure_out = 1.629*plasma_protein_concentration_out+0.2935*(plasma_protein_concentration_out**2)
oncotic_pressure_avg = (Oncotic_pressure_in+Oncotic_pressure_out)/2
Na_concentration = (sodium_amount / extracellular_fluid_volume)
Na_water_controller = Na_controller_gain*(Kp_VP*(Na_concentration - ref_Na_concentration)+Ki_VP*Na_concentration_error)
normalized_vasopressin_concentration = 1 + Na_water_controller
vasopressin_concentration = nominal_vasopressin_conc * normalized_vasopressin_concentration
water_intake_vasopressin_int = 1-water_intake_vasopressin_scale/2
water_intake = water_intake_species_scale*(nom_water_intake/60/24)*(water_intake_vasopressin_int + water_intake_vasopressin_scale/(1+safe_exp((normalized_vasopressin_concentration-1)/water_intake_vasopressin_slope)))
daily_water_intake = (water_intake * 24 * 60)
L_pt_s1 = L_pt_s1_nom*(1+tubular_length_increase)
L_pt_s2 = L_pt_s2_nom*(1+tubular_length_increase)
L_pt_s3 = L_pt_s3_nom*(1+tubular_length_increase)
Dc_pt = Dc_pt_nom*(1+tubular_diameter_increase)
L_pt = L_pt_s1+L_pt_s2 + L_pt_s3
SN_filtered_Na_load = (SNGFR_nL_min / 1000 / 1000000)*Na_concentration
filtered_Na_load = SN_filtered_Na_load*number_of_functional_nephrons
pressure_natriuresis_PT_int = 1 - pressure_natriuresis_PT_scale/2
pressure_natriuresis_PT_effect = max_fn(0.001,pressure_natriuresis_PT_int + pressure_natriuresis_PT_scale / (1 + safe_exp((mean_arterial_pressure_MAP- nominal_map_setpoint) / pressure_natriuresis_PT_slope)))
pressure_natriuresis_LoH_int = 1 - pressure_natriuresis_LoH_scale/2
pressure_natriuresis_LoH_effect = max_fn(0.001,pressure_natriuresis_LoH_int + pressure_natriuresis_LoH_scale / (1 + safe_exp((mean_arterial_pressure_MAP - nominal_map_setpoint) / pressure_natriuresis_LoH_slope)))
pressure_natriuresis_DCT_magnitude = max_fn(0,pressure_natriuresis_DCT_scale )
pressure_natriuresis_DCT_int = 1 - pressure_natriuresis_DCT_magnitude/2
pressure_natriuresis_DCT_effect = max_fn(0.001,pressure_natriuresis_DCT_int + pressure_natriuresis_DCT_magnitude/ (1 + safe_exp((mean_arterial_pressure_MAP - nominal_map_setpoint) / pressure_natriuresis_DCT_slope)))
pressure_natriuresis_CD_magnitude = max_fn(0,pressure_natriuresis_CD_scale *(1+disease_effects_decreasing_CD_PN))
pressure_natriuresis_CD_int = 1 - pressure_natriuresis_CD_magnitude/2
pressure_natriuresis_CD_effect = max_fn(0.001,pressure_natriuresis_CD_int + pressure_natriuresis_CD_magnitude/ (1 + safe_exp((mean_arterial_pressure_MAP - nominal_map_setpoint) / pressure_natriuresis_CD_slope)))
AT1_PT_int = 1 - AT1_PT_slope*nominal_equilibrium_AT1_bound_AngII
g_at = AT1_PT_int + AT1_PT_slope*AT1_bound_AngII
g_rsna_int = 1-pt_rsna_scale/2
g_rsna = g_rsna_int + pt_rsna_scale/ (1 + safe_exp((1 - renal_sympathetic_nerve_activity) / pt_rsna_slope))
e_pt_sodreab = nominal_pt_na_reabsorption * g_at * g_rsna*pressure_natriuresis_PT_effect*CA_inhibitor
'''
SYSTEMIC_KERNEL = compile(SYSTEMIC_SOURCE, '<systemic_kernel>', 'exec')
print('SYSTEMIC: 100 algebraic equations')


In [ ]:
PROXIMAL_SOURCE = r'''
glucose_reabs_per_unit_length_s1 = nom_glucose_reabs_per_unit_length_s1*diabetic_adaptation*SGLT2_inhibition
glucose_reabs_per_unit_length_s2 = nom_glucose_reabs_per_unit_length_s2*diabetic_adaptation*SGLT2_inhibition
glucose_reabs_per_unit_length_s3 = nom_glucose_reabs_per_unit_length_s3*diabetic_adaptation*SGLT1_inhibition
SN_filtered_glucose_load = glucose_concentration*SNGFR_nL_min / 1000 / 1000000
glucose_pt_out_s1 = max_fn(0,SN_filtered_glucose_load-glucose_reabs_per_unit_length_s1*L_pt_s1)
glucose_pt_out_s2 = max_fn(0,glucose_pt_out_s1-glucose_reabs_per_unit_length_s2*L_pt_s2)
glucose_pt_out_s3 = max_fn(0,glucose_pt_out_s2-glucose_reabs_per_unit_length_s3*L_pt_s3)
uge = glucose_pt_out_s3*2e6*24*60*180/1000
RUGE = glucose_pt_out_s3*number_of_functional_nephrons*180
SN_filtered_Na_load = (SNGFR_nL_min / 1000 / 1000000)*Na_concentration
SGTL2_Na_reabs_mmol_s1 = SN_filtered_glucose_load-glucose_pt_out_s1
SGTL2_Na_reabs_mmol_s2 = glucose_pt_out_s1-glucose_pt_out_s2
SGTL1_Na_reabs_mmol = 2*(glucose_pt_out_s2-glucose_pt_out_s3)
total_SGLT2_Na_reabs = SGTL2_Na_reabs_mmol_s1+SGTL2_Na_reabs_mmol_s2+SGTL1_Na_reabs_mmol
e_pt_sodreab_adj = e_pt_sodreab - 0.039
Na_reabs_per_unit_length = -safe_log(1-e_pt_sodreab)/(L_pt_s1+L_pt_s2+L_pt_s3)
Na_pt_out_s1 = SN_filtered_Na_load*safe_exp(-Na_reabs_per_unit_length*L_pt_s1) - SGTL2_Na_reabs_mmol_s1
Na_pt_out_s2 = Na_pt_out_s1*safe_exp(-Na_reabs_per_unit_length*L_pt_s2) - SGTL2_Na_reabs_mmol_s2
Na_pt_out_s3 = Na_pt_out_s2*safe_exp(-Na_reabs_per_unit_length*L_pt_s3) - SGTL1_Na_reabs_mmol
PT_Na_reabs_fraction = 1-Na_pt_out_s3/SN_filtered_Na_load
SN_filtered_urea_load = (SNGFR_nL_min / 1000 / 1000000)*plasma_urea
urea_out_s1 = SN_filtered_urea_load - urea_permeability_PT*(SN_filtered_urea_load/(0.5*((SNGFR_nL_min / 1000 / 1000000)+water_out_s1_delayed))-plasma_urea)*water_out_s1_delayed
urea_out_s2 = urea_out_s1 - urea_permeability_PT*(urea_out_s1/(0.5*(water_out_s1_delayed+water_out_s2_delayed))-plasma_urea)*water_out_s2_delayed
urea_out_s3 = urea_out_s2 - urea_permeability_PT*(urea_out_s2/(0.5*(water_out_s2_delayed+water_out_s3_delayed))-plasma_urea)*water_out_s3_delayed
urea_reabsorption_fraction = 1-urea_out_s3/SN_filtered_urea_load
osmoles_out_s1 = 2*Na_pt_out_s1 + glucose_pt_out_s1 + urea_out_s1
water_out_s1 = (((SNGFR_nL_min / 1000 / 1000000)/(2*SN_filtered_Na_load+SN_filtered_glucose_load+ SN_filtered_urea_load)))*osmoles_out_s1
osmoles_out_s2 = 2*Na_pt_out_s2 + glucose_pt_out_s2 + urea_out_s2
water_out_s2 = (water_out_s1/osmoles_out_s1)*osmoles_out_s2
osmoles_out_s3 = 2*Na_pt_out_s3 + glucose_pt_out_s3 + urea_out_s3
water_out_s3 = (water_out_s2/osmoles_out_s2)*osmoles_out_s3
PT_water_reabs_fraction = 1-water_out_s3/(SNGFR_nL_min / 1000 / 1000000)
Na_concentration_out_s1 = Na_pt_out_s1/water_out_s1
Na_concentration_out_s2 = Na_pt_out_s2/water_out_s2
Na_concentration_out_s3 = Na_pt_out_s3/water_out_s3
glucose_concentration_out_s1 = glucose_pt_out_s1/water_out_s1
glucose_concentration_out_s2 = glucose_pt_out_s2/water_out_s2
glucose_concentration_out_s3 = glucose_pt_out_s3/water_out_s3
urea_concentration_out_s1 = urea_out_s1/water_out_s1
urea_concentration_out_s2 = urea_out_s2/water_out_s2
urea_concentration_out_s3 = urea_out_s3/water_out_s3
osmolality_out_s1 = osmoles_out_s1/water_out_s1
osmolality_out_s2 = osmoles_out_s1/water_out_s2
osmolality_out_s3 = osmoles_out_s1/water_out_s3
PT_Na_outflow = Na_pt_out_s3*number_of_functional_nephrons
PT_Na_reab_perUnitSA = SN_filtered_Na_load*e_pt_sodreab/(3.14*Dc_pt*(L_pt_s1+L_pt_s2+L_pt_s3))
normalized_PT_reabsorption_density = PT_Na_reab_perUnitSA/PT_Na_reab_perUnitSA_0
PT_Na_reabs_effect_increasing_tubular_length = 0
PT_Na_reabs_effect_increasing_tubular_diameter = 0
'''
PROXIMAL_KERNEL = compile(PROXIMAL_SOURCE, '<proximal_kernel>', 'exec')
print('PROXIMAL: 49 algebraic equations')


In [ ]:
LOOP_TGF_SOURCE = r'''
water_in_DescLoH = water_out_s3
Na_in_DescLoH = Na_pt_out_s3
urea_in_DescLoH = urea_out_s3
glucose_in_DescLoH = glucose_pt_out_s3
osmoles_in_DescLoH = osmoles_out_s3
Na_concentration_in_DescLoH = Na_concentration_out_s3
Urea_concentration_in_DescLoH = urea_concentration_out_s3
glucose_concentration_in_DescLoH = glucose_concentration_out_s3
osmolality_in_DescLoH = osmoles_out_s3/water_out_s3
Na_out_DescLoH = Na_in_DescLoH
urea_out_DescLoH = urea_in_DescLoH
glucose_out_DescLoH = glucose_in_DescLoH
osmoles_out_DescLoH = osmoles_in_DescLoH
deltaLoH_NaFlow = LoH_flow_dependence*(Na_out_DescLoH-nom_Na_in_AscLoH)
AscLoH_Reab_Rate = (2*nominal_loh_na_reabsorption*(nom_Na_in_AscLoH+deltaLoH_NaFlow))/L_lh_des
effective_AscLoH_Reab_Rate = AscLoH_Reab_Rate*pressure_natriuresis_LoH_effect
osmolality_out_DescLoH = osmolality_in_DescLoH*safe_exp(min_fn(effective_AscLoH_Reab_Rate*L_lh_des,2*Na_in_DescLoH)/(water_in_DescLoH*osmolality_in_DescLoH))
water_out_DescLoH = water_in_DescLoH*osmolality_in_DescLoH/osmolality_out_DescLoH
Na_concentration_out_DescLoH = Na_out_DescLoH/water_out_DescLoH
glucose_concentration_out_DescLoH = glucose_out_DescLoH/water_out_DescLoH
urea_concentration_out_DescLoH = urea_out_DescLoH/water_out_DescLoH
Na_in_AscLoH = Na_out_DescLoH
urea_in_AscLoH_before_secretion = urea_out_DescLoH
glucose_in_AscLoH = glucose_out_DescLoH
osmoles_in_AscLoH_before_secretion = osmoles_out_DescLoH
water_in_AscLoH = water_out_DescLoH
urea_in_AscLoH = urea_in_AscLoH_before_secretion + reabsorbed_urea_cd_delayed
urea_concentration_in_AscLoH = urea_in_AscLoH/water_out_DescLoH
osmoles_in_AscLoH = osmoles_in_AscLoH_before_secretion  + reabsorbed_urea_cd_delayed
osmolality_in_AscLoH = osmoles_in_AscLoH/water_in_AscLoH
osmolality_out_AscLoH = osmolality_in_AscLoH - min_fn(L_lh_des*effective_AscLoH_Reab_Rate, 2*Na_in_DescLoH)*(safe_exp(min_fn(L_lh_des*effective_AscLoH_Reab_Rate, 2*Na_in_DescLoH)/(water_in_DescLoH*osmolality_in_DescLoH))/water_in_DescLoH)
osmoles_reabsorbed_AscLoH = (osmolality_in_AscLoH - osmolality_out_AscLoH)*water_in_AscLoH
Na_reabsorbed_AscLoH = osmoles_reabsorbed_AscLoH/2
Na_out_AscLoH = max_fn(0,Na_in_AscLoH - Na_reabsorbed_AscLoH)
urea_out_AscLoH = urea_in_AscLoH
glucose_out_AscLoH = glucose_in_AscLoH
water_out_AscLoH = water_in_AscLoH
osmoles_out_AscLoH = osmolality_out_AscLoH*water_out_AscLoH
Na_concentration_out_AscLoH = Na_out_AscLoH/water_out_AscLoH
glucose_concentration_out_AscLoH = glucose_out_AscLoH/water_out_AscLoH
urea_concentration_out_AscLoH = urea_out_AscLoH/water_out_AscLoH
LoH_reabs_fraction = 1-Na_out_AscLoH/Na_in_AscLoH
SN_macula_densa_Na_flow = Na_out_AscLoH
MD_Na_concentration = Na_concentration_out_AscLoH
TGF0_tubulo_glomerular_feedback = 1 - S_tubulo_glomerular_feedback/2
tubulo_glomerular_feedback_signal = (TGF0_tubulo_glomerular_feedback + S_tubulo_glomerular_feedback / (1 + safe_exp((MD_Na_concentration_setpoint - MD_Na_concentration)/ F_md_scale_tubulo_glomerular_feedback)))
psi_al = scale_aldo_DT * Aldo_MR_normalised_effect ** hill_aldo_DT
e_dct_sodreab = nominal_dt_na_reabsorption * psi_al*HCTZ_effect_on_DT_Na_reabs*pressure_natriuresis_DCT_effect
'''
LOOP_TGF_KERNEL = compile(LOOP_TGF_SOURCE, '<loop_tgf_kernel>', 'exec')
print('LOOP_TGF: 48 algebraic equations')


In [ ]:
DISTAL_SOURCE = r'''
water_in_DCT = water_out_AscLoH
Na_in_DCT = Na_out_AscLoH
urea_in_DCT = urea_out_AscLoH
glucose_in_DCT = glucose_out_AscLoH
osmoles_in_DCT = osmoles_out_AscLoH
Na_concentration_in_DCT = Na_concentration_out_AscLoH
urea_concentration_in_DCT = urea_concentration_out_AscLoH
glucose_concentration_in_DCT = glucose_concentration_out_AscLoH
osmolality_in_DCT = osmolality_out_AscLoH
urea_out_DCT = urea_in_DCT
glucose_out_DCT = glucose_in_DCT
water_out_DCT = water_in_DCT
urea_concentration_out_DCT = urea_out_DCT/water_out_DCT
glucose_concentration_out_DCT = glucose_out_DCT/water_out_DCT
deltaDCT_NaFlow = DCT_flow_dependence*(Na_in_DCT-nom_Na_in_DCT)
e_dct_sodreab_adj = max_fn(0, min_fn(1,(e_dct_sodreab*(nom_Na_in_DCT+deltaDCT_NaFlow))/Na_in_DCT))
R_dct = -safe_log(1-e_dct_sodreab_adj)/L_dct
Na_out_DCT = Na_in_DCT*safe_exp(-R_dct*L_dct)
Na_concentration_out_DCT = Na_out_DCT/water_out_DCT
osmolality_out_DCT = 2*Na_concentration_out_DCT + glucose_concentration_out_DescLoH + urea_concentration_in_AscLoH
osmoles_out_DCT = osmolality_out_DCT*water_out_DCT
DCT_Na_reabs_fraction = 1-Na_out_DCT/Na_in_DCT
lambda_anp = 1 - anp_cd_slope * (normalized_atrial_NP_concentration - 1)
lambda_aldo_CD = scale_aldo_CD*((Aldo_MR_normalised_effect)**hill_aldo_CD)
e_cd_sodreab = min_fn(0.99,nominal_cd_na_reabsorption * lambda_anp *lambda_aldo_CD*pressure_natriuresis_CD_effect)
'''
DISTAL_KERNEL = compile(DISTAL_SOURCE, '<distal_kernel>', 'exec')
print('DISTAL: 25 algebraic equations')


In [ ]:
COLLECTING_SOURCE = r'''
water_in_CD = water_out_DCT
Na_in_CD = Na_out_DCT
urea_in_CD = urea_out_DCT
glucose_in_CD = glucose_out_DCT
osmoles_in_CD = osmoles_out_DCT
Na_concentration_in_CD = Na_concentration_out_DCT
urea_concentration_in_CD = urea_concentration_out_DCT
glucose_concentration_in_CD = glucose_concentration_out_DCT
osmolality_in_CD = osmolality_out_DCT
deltaCD_NaFlow = CD_flow_dependence*(Na_in_CD-nom_Na_in_CD)
e_cd_sodreab_adj = max_fn(0,min_fn(1,(e_cd_sodreab*(nom_Na_in_CD+deltaCD_NaFlow))/Na_in_CD))
R_cd = -safe_log(1-e_cd_sodreab_adj)/L_cd
Na_out_CD = Na_in_CD*safe_exp(-R_cd*L_cd)
CD_Na_reabs_fraction = 1-Na_out_CD/Na_in_CD
ADH_water_permeability = min_fn(1,max_fn(0,nom_ADH_water_permeability*normalized_vasopressin_concentration))
ADH_urea_permeability = min_fn(1,max_fn(0,nom_ADH_urea_permeability*(1+ (normalized_vasopressin_concentration-1)/5)))
max_water_reabs_CD = water_in_CD-(osmolality_in_CD*water_in_CD-2*Na_in_CD*(1-safe_exp(-R_cd*L_cd)))/osmolality_in_AscLoH
water_out_CD_before_urea_reabs = max_fn(0,water_in_CD - ADH_water_permeability*max_water_reabs_CD)
osmoles_out_CD_before_urea_reabsorption = osmoles_in_CD-2*(Na_in_CD - Na_out_CD)
osmolality_out_CD_before_urea_reabsorption = osmoles_out_CD_before_urea_reabsorption/water_out_CD_before_urea_reabs
urea_concentration_out_CD_before_reabsorption = urea_in_CD/water_out_CD_before_urea_reabs
reabsorbed_urea_cd = ADH_urea_permeability*(urea_concentration_out_CD_before_reabsorption - urea_concentration_out_DescLoH)*water_out_CD_before_urea_reabs
urea_out_CD = urea_in_CD - reabsorbed_urea_cd
urea_concentration_out_CD = urea_out_CD/water_out_CD_before_urea_reabs
osmoles_out_CD_after_urea_reabsorption = osmoles_out_CD_before_urea_reabsorption - reabsorbed_urea_cd
water_reabsorption_CD_after_urea_reabsorption = ADH_water_permeability*(water_out_CD_before_urea_reabs - osmoles_out_CD_after_urea_reabsorption/osmolality_in_AscLoH)
water_out_CD_after_urea_reabsorption = max_fn(0,water_out_CD_before_urea_reabs - water_reabsorption_CD_after_urea_reabsorption)
osmolality_out_CD_after_urea_reabsorption = osmoles_out_CD_after_urea_reabsorption/water_out_CD_after_urea_reabsorption
min_urine_flow_rate = (water_in_CD-max_water_reabs_CD - water_reabsorption_CD_after_urea_reabsorption/ADH_water_permeability)*number_of_functional_nephrons*60*24
urine_flow_rate = water_out_CD_after_urea_reabsorption*number_of_functional_nephrons
daily_urine_flow = (urine_flow_rate * 60 * 24)
Na_excretion_via_urine = Na_out_CD*number_of_functional_nephrons
Na_balance = Na_intake_rate - Na_excretion_via_urine
water_balance = daily_water_intake - daily_urine_flow
FENA = Na_excretion_via_urine/filtered_Na_load
PT_fractional_glucose_reabs = (SN_filtered_glucose_load - glucose_pt_out_s3)/SN_filtered_glucose_load
PT_fractional_Na_reabs = (SN_filtered_Na_load - Na_pt_out_s3)/SN_filtered_Na_load
PT_fractional_urea_reabs = ( SN_filtered_urea_load - urea_out_s3)/SN_filtered_urea_load
PT_fractional_water_reabs = ((SNGFR_nL_min / 1000 / 1000000) - water_out_s3)/(SNGFR_nL_min / 1000 / 1000000)
LoH_fractional_Na_reabs = (Na_in_DescLoH - Na_out_AscLoH)/Na_in_DescLoH
LoH_fractional_water_reabs = (water_in_DescLoH - water_out_AscLoH)/water_in_DescLoH
DCT_fractional_Na_reabs = (Na_in_DCT - Na_out_DCT)/Na_in_DCT
CD_fractional_Na_reabs = (Na_in_CD - Na_out_CD)/Na_in_CD
CD_fractional_urea_reabs = (urea_in_CD - urea_out_CD)/urea_in_CD
CD_OM_fractional_water_reabs = (water_in_CD - water_out_CD_before_urea_reabs)/water_in_CD
CD_IM_fractional_water_reabs = (water_out_CD_before_urea_reabs - water_out_CD_after_urea_reabsorption)/water_out_CD_before_urea_reabs
Oncotic_pressure_peritubular_in = Oncotic_pressure_out
plasma_protein_concentration_peritubular_out = (SNRBF_nl_min)*plasma_protein_concentration/(SNRBF_nl_min-urine_flow_rate*1e6*1000/number_of_functional_nephrons)
Oncotic_pressure_peritubular_out = 1.629*plasma_protein_concentration_peritubular_out+0.2935*(plasma_protein_concentration_peritubular_out**2)
oncotic_pressure_peritubular_avg = (Oncotic_pressure_peritubular_in+Oncotic_pressure_peritubular_out)/2
tubular_reabsorption = GFR_ml_min/1000 - urine_flow_rate
RIHP = postglomerular_pressure - (oncotic_pressure_peritubular_avg - interstitial_oncotic_pressure) + tubular_reabsorption/nom_peritubular_cap_Kf
'''
COLLECTING_KERNEL = compile(COLLECTING_SOURCE, '<collecting_kernel>', 'exec')
print('COLLECTING: 52 algebraic equations')


In [ ]:
PRESSURE_SOURCE = r'''
mmHg_Nperm2_conv = 133.32
Pc_pt_s1 = Pc_pt_s1_mmHg*mmHg_Nperm2_conv
Pc_pt_s2 = Pc_pt_s2_mmHg*mmHg_Nperm2_conv
Pc_pt_s3 = Pc_pt_s3_mmHg*mmHg_Nperm2_conv
Pc_lh_des = Pc_lh_des_mmHg*mmHg_Nperm2_conv
Pc_lh_asc = Pc_lh_asc_mmHg*mmHg_Nperm2_conv
Pc_dt = Pc_dt_mmHg*mmHg_Nperm2_conv
Pc_cd = Pc_cd_mmHg*mmHg_Nperm2_conv
P_interstitial = 4.9*mmHg_Nperm2_conv
pi = 3.14
B1 = (4*tubular_compliance+1)*128*gamma/pi
mean_cd_water_flow = (water_in_CD-water_out_CD_after_urea_reabsorption)/2
B2_cd = (Pc_cd**(4*tubular_compliance))/(Dc_cd**4)
P_in_cd = (0**(4*tubular_compliance+1)+B1*B2_cd*(mean_cd_water_flow/1e3)*L_cd)**(1/(4*tubular_compliance+1))
P_in_cd_mmHg = (P_in_cd+P_interstitial)/mmHg_Nperm2_conv
B2_dt = (Pc_dt**(4*tubular_compliance))/(Dc_dt**4)
P_in_dt = (P_in_cd**(4*tubular_compliance+1)+B1*B2_dt*(water_in_DCT/1e3)*L_dct)**(1/(4*tubular_compliance+1))
P_in_dt_mmHg = (P_in_dt+P_interstitial)/mmHg_Nperm2_conv
B2_lh_asc = (Pc_lh_asc**(4*tubular_compliance))/(Dc_lh**4)
P_in_lh_asc = (P_in_dt**(4*tubular_compliance+1)+B1*B2_lh_asc*(water_in_AscLoH/1e3)*L_lh_asc)**(1/(4*tubular_compliance+1))
P_in_lh_asc_mmHg = (P_in_lh_asc+P_interstitial)/mmHg_Nperm2_conv
A_lh_des = effective_AscLoH_Reab_Rate/(water_in_DescLoH*osmolality_in_DescLoH)
B2_lh_des = (Pc_lh_des**(4*tubular_compliance))*(water_in_DescLoH/1e3)/((Dc_lh**4)*A_lh_des)
P_in_lh_des = (P_in_lh_asc**(4*tubular_compliance+1)+B1*B2_lh_des*(1-safe_exp(-A_lh_des*L_lh_des)))**(1/(4*tubular_compliance+1))
P_in_lh_des_mmHg = (P_in_lh_des+P_interstitial)/mmHg_Nperm2_conv
Rurea = (SN_filtered_urea_load - urea_out_s3)/(L_pt_s1+L_pt_s2+L_pt_s3)
urea_in_s2 = SN_filtered_urea_load - Rurea*L_pt_s1
urea_in_s3 = SN_filtered_urea_load - Rurea*(L_pt_s1+L_pt_s2)
A_na = Na_reabs_per_unit_length
flow_integral_s3 = 2*(Na_pt_out_s2/A_na)*(1-safe_exp(-A_na*L_pt_s3)) - (3/2)*glucose_pt_out_s2*L_pt_s3**2 + urea_in_s3*L_pt_s3 - (1/2)*Rurea*(L_pt_s3**2)
flow_integral_s2 = 2*(Na_pt_out_s1/A_na)*(1-safe_exp(-A_na*L_pt_s2)) - (1/2)*glucose_pt_out_s1*L_pt_s2**2 + urea_in_s2*L_pt_s2 - (1/2)*Rurea*(L_pt_s2**2)
flow_integral_s1 = 2*(SN_filtered_Na_load/A_na)*(1-safe_exp(-A_na*L_pt_s1)) - (1/2)*SN_filtered_glucose_load*L_pt_s1**2 + SN_filtered_urea_load*L_pt_s1 - (1/2)*Rurea*(L_pt_s1**2)
B2_pt_s3 = (Pc_pt_s3**(4*tubular_compliance))/(Dc_pt**4)
B3_pt_s3 = (water_out_s2/1e3)/osmoles_out_s2
P_in_pt_s3 = (P_in_lh_des**(4*tubular_compliance+1)+B1*B2_pt_s3*B3_pt_s3*flow_integral_s3)**(1/(4*tubular_compliance+1))
P_in_pt_s3_mmHg = (P_in_pt_s3+P_interstitial)/mmHg_Nperm2_conv
B2_pt_s2 = (Pc_pt_s3**(4*tubular_compliance))/(Dc_pt**4)
B3_pt_s2 = (water_out_s1/1e3)/osmoles_out_s1
P_in_pt_s2 = (P_in_pt_s3**(4*tubular_compliance+1)+B1*B2_pt_s2*B3_pt_s2*flow_integral_s2)**(1/(4*tubular_compliance+1))
P_in_pt_s2_mmHg = (P_in_pt_s2+P_interstitial)/mmHg_Nperm2_conv
B2_pt_s1 = (Pc_pt_s1**(4*tubular_compliance))/(Dc_pt**4)
B3_pt_s1 = (SNGFR_nL_min / 1e12)/(2*SN_filtered_Na_load+SN_filtered_glucose_load+ SN_filtered_urea_load)
P_in_pt_s1 = (P_in_pt_s2**(4*tubular_compliance+1)+B1*B2_pt_s1*B3_pt_s1*flow_integral_s1)**(1/(4*tubular_compliance+1))
P_in_pt_s1_mmHg = (P_in_pt_s1+P_interstitial)/mmHg_Nperm2_conv
'''
PRESSURE_KERNEL = compile(PRESSURE_SOURCE, '<pressure_kernel>', 'exec')
print('PRESSURE: 44 algebraic equations')


In [ ]:
RAAS_SOURCE = r'''
K_Na_ratio_effect_on_aldo = max_fn(((potassium_concentration / Na_concentration) / 0.0034722) - 9.29,0)
AT1_aldo_int = 1 - AT1_aldo_slope*nominal_equilibrium_AT1_bound_AngII
AngII_effect_on_aldo = AT1_aldo_int + AT1_aldo_slope*AT1_bound_AngII
N_als = (K_Na_ratio_effect_on_aldo * AngII_effect_on_aldo )
rsna_renin_intercept = 1-rsna_renin_slope
rnsa_effect_on_renin_secretion = rsna_renin_slope * renal_sympathetic_nerve_activity + rsna_renin_intercept
md_effect_on_renin_secretion = md_renin_A*safe_exp(-md_renin_tau*(SN_macula_densa_Na_flow*baseline_nephrons - nom_LoH_Na_outflow))
AT1_bound_AngII_effect_on_PRA = (10 ** (AT1_PRC_slope * safe_log10(AT1_bound_AngII / nominal_equilibrium_AT1_bound_AngII) + AT1_PRC_yint))
plasma_renin_activity = concentration_to_renin_activity_conversion_plasma* plasma_renin_concentration*DRI_effect_on_PRA
renin_secretion_rate = (safe_log(2)/renin_half_life)*nominal_equilibrium_PRC*AT1_bound_AngII_effect_on_PRA*md_effect_on_renin_secretion*rnsa_effect_on_renin_secretion*HCTZ_effect_on_renin_secretion
renin_degradation_rate = safe_log(2)/renin_half_life
AngI_degradation_rate = safe_log(2)/AngI_half_life
AngII_degradation_rate = safe_log(2)/AngII_half_life
AT1_bound_AngII_degradation_rate = safe_log(2)/AT1_bound_AngII_half_life
AT2_bound_AngII_degradation_rate = safe_log(2)/AT2_bound_AngII_half_life
ACE_activity = nominal_ACE_activity
chymase_activity = nominal_chymase_activity
AT1_receptor_binding_rate = nominal_AT1_receptor_binding_rate
AT2_receptor_binding_rate = nominal_AT2_receptor_binding_rate
'''
RAAS_KERNEL = compile(RAAS_SOURCE, '<raas_kernel>', 'exec')
print('RAAS: 19 algebraic equations')


In [ ]:
KERNELS = [SYSTEMIC_KERNEL, PROXIMAL_KERNEL, LOOP_TGF_KERNEL, DISTAL_KERNEL, COLLECTING_KERNEL, PRESSURE_KERNEL, RAAS_KERNEL]

def evaluate_context(t, y, parameters):
    context = dict(parameters)
    context['time'] = float(t)
    context.update(zip(STATES, np.asarray(y, dtype=float)))
    for kernel in KERNELS:
        run_kernel(context, kernel)
    return context

DERIVATIVE_SOURCE = r'''
dy[0] = plasma_renin_activity - (AngI) * (chymase_activity + ACE_activity) - (AngI) * AngI_degradation_rate
dy[1] = AngI * (chymase_activity + ACE_activity) - AngII * AngII_degradation_rate - AngII*AT1_receptor_binding_rate - AngII* (AT2_receptor_binding_rate)
dy[2] = AngII * (AT1_receptor_binding_rate) - AT1_bound_AngII_degradation_rate*AT1_bound_AngII
dy[3] = AngII * (AT2_receptor_binding_rate) - AT2_bound_AngII_degradation_rate*AT2_bound_AngII
dy[4] = renin_secretion_rate - plasma_renin_concentration * renin_degradation_rate
dy[5] = C_prerenal_blood_pressure * (mean_arterial_pressure_MAP - pre_renal_blood_pressure)
dy[6] = C_water_intake_ecf_volume * (water_intake) + C_urine_flow_ecf_volume * (urine_flow_rate)
dy[7] = C_na_excretion_na_amount * (Na_excretion_via_urine) + C_na_intake_na_amount * (Na_intake_rate)
dy[8] = C_tgf * (tubulo_glomerular_feedback_signal-tubulo_glomerular_feedback_effect)
dy[9] = C_aldo_secretion * (N_als-normalized_aldosterone_level)
dy[10] = 500*(preafferent_pressure_autoreg_function - preafferent_pressure_autoreg_signal)
dy[11] = 500*(glomerular_pressure_autoreg_function - glomerular_pressure_autoreg_signal)
dy[12] = 0
dy[13] = C_cardiac_output_delayed*(cardiac_output - cardiac_output_delayed)
dy[14] = C_co_error*(cardiac_output-CO_nom)
dy[15] = C_Na_error*(Na_concentration - ref_Na_concentration)
dy[16] = C_tgf_reset*(SN_macula_densa_Na_flow*baseline_nephrons - F0_TGF)
dy[17] = C_P_bowmans*(P_in_pt_s1_mmHg - P_bowmans)
dy[18] = C_P_oncotic*(oncotic_pressure_avg - oncotic_pressure_difference)
dy[19] = C_rbf*(renal_blood_flow_L_min - renal_blood_flow_L_min_delayed)
dy[20] = C_rihp*(RIHP - renal_interstitial_hydrostatic_pressure)
dy[21] = GP_effect_increasing_Kf
dy[22] = CD_PN_loss_rate
dy[23] = PT_Na_reabs_effect_increasing_tubular_length
dy[24] = PT_Na_reabs_effect_increasing_tubular_diameter
dy[25] = C_pt_water*(water_out_s1 - water_out_s1_delayed)
dy[26] = C_pt_water*(water_out_s2 - water_out_s2_delayed)
dy[27] = C_pt_water*(water_out_s3 - water_out_s3_delayed)
dy[28] = C_pt_water*(reabsorbed_urea_cd - reabsorbed_urea_cd_delayed)
dy[29] = RUGE
dy[30] = urine_flow_rate
'''
DERIVATIVE_KERNEL = compile(DERIVATIVE_SOURCE, '<31_state_rhs>', 'exec')

def renal_rhs(t, y, parameters):
    context = evaluate_context(t, y, parameters)
    context['dy'] = np.empty(len(STATES), dtype=float)
    run_kernel(context, DERIVATIVE_KERNEL)
    return context['dy']

print('ODE assembled:', len(STATES), 'derivatives')


In [ ]:
def nominal_initial_conditions(p=PARAMS):
    values = {
        'AngI': 8.164, 'AngII': 5.17, 'AT1_bound_AngII': 16.6,
        'AT2_bound_AngII': 5.5, 'plasma_renin_concentration': 17.845,
        'pre_renal_blood_pressure': p['nominal_map_setpoint'],
        'extracellular_fluid_volume': p['ECF_nom'],
        'sodium_amount': p['ECF_nom'] * p['ref_Na_concentration'],
        'tubulo_glomerular_feedback_effect': 1.0, 'normalized_aldosterone_level': 1.0,
        'preafferent_pressure_autoreg_signal': 1.0, 'glomerular_pressure_autoreg_signal': 1.0,
        'F_out_dt_delay': 5e-12, 'cardiac_output_delayed': p['CO_nom'],
        'CO_error': 0.0, 'Na_concentration_error': 0.0,
        'F0_TGF': p['nom_LoH_Na_outflow'], 'P_bowmans': p['Pc_pt_s1_mmHg'],
        'oncotic_pressure_difference': p['nom_oncotic_pressure_difference'],
        'renal_blood_flow_L_min_delayed': p['nom_renal_blood_flow_L_min'],
        'renal_interstitial_hydrostatic_pressure': p['RIHP0'],
        'disease_effects_increasing_Kf': 0.0, 'disease_effects_decreasing_CD_PN': 0.0,
        'tubular_length_increase': 0.0, 'tubular_diameter_increase': 0.0,
        'water_out_s1_delayed': 3e-8, 'water_out_s2_delayed': 1.9e-8,
        'water_out_s3_delayed': 1.2e-8, 'reabsorbed_urea_cd_delayed': 2e-8,
        'UGE': 0.0, 'cumWaterExcretion': 0.0,
    }
    return np.array([values[name] for name in STATES], dtype=float)

def load_r_steady_warm_start():
    path = R_DATA_DIR / 'steady_state_initial_conditions.csv'
    if not path.exists():
        return None
    frame = pd.read_csv(path).set_index('state')['value']
    missing = [name for name in STATES if name not in frame.index]
    if missing:
        raise ValueError(f'R steady-state file is missing states: {missing}')
    return frame.loc[STATES].to_numpy(dtype=float)

def continue_to_steady_state(y0, parameters, block_minutes=5000.0, max_blocks=200,
                             state_rtol=1e-8, state_atol=1e-10):
    current = np.asarray(y0, dtype=float)
    dynamic = np.array([i for i, s in enumerate(STATES) if s not in {'UGE', 'cumWaterExcretion'}])
    history = []
    for block in range(1, max_blocks + 1):
        sol = solve_ivp(lambda t, y: renal_rhs(t, y, parameters), (0.0, block_minutes), current,
                        method='LSODA', rtol=1e-8, atol=1e-10, max_step=10.0)
        if not sol.success:
            raise RuntimeError(sol.message)
        nxt = sol.y[:, -1]
        scale = state_atol + state_rtol * np.maximum(np.abs(nxt[dynamic]), np.abs(current[dynamic]))
        ratio = float(np.max(np.abs(nxt[dynamic] - current[dynamic]) / scale))
        history.append({'block': block, 'time_total_min': block * block_minutes,
                        'state_max_scaled_change': ratio})
        current = nxt
        if block >= 5 and ratio <= 1.0:
            return current, pd.DataFrame(history)
    warnings.warn('Steady-state continuation reached max_blocks before the strict threshold.')
    return current, pd.DataFrame(history)


In [ ]:
def build_gfr_parameters(input_params=None, *, tgf_strength=0.7, ca_inhibitor=1.0):
    """Return model parameters for one GFR run.

    `input_params` may be a dict keyed by the 86 names or a vector in
    `GSA_INPUT_NAMES` order. Non-GSA parameters remain fixed.
    """
    p = apply_figure2_configuration(PARAMS, tgf_strength=tgf_strength, ca_inhibitor=ca_inhibitor)
    if input_params is None:
        updates = GSA_INPUT_BASELINE
    elif isinstance(input_params, dict):
        updates = input_params
    else:
        values = list(input_params)
        if len(values) != GSA_INPUT_DIMENSION:
            raise ValueError(f'Expected {GSA_INPUT_DIMENSION} inputs, got {len(values)}')
        updates = dict(zip(GSA_INPUT_NAMES, values))
    unknown = sorted(set(updates) - set(GSA_INPUT_NAMES))
    if unknown:
        raise KeyError(f'Unknown GSA input parameter(s): {unknown}')
    p.update({name: float(value) for name, value in updates.items()})
    return p

def run_gfr_model(input_params=None, *, y0=None, return_details=False):
    """Run the Hallow 2017 model to steady state and return GFR_ml_min."""
    p = build_gfr_parameters(input_params)
    if y0 is None:
        y0 = nominal_initial_conditions(p)
    steady_state, convergence = continue_to_steady_state(y0, p)
    ctx = evaluate_context(0.0, steady_state, p)
    result = {
        'GFR_ml_min': float(ctx['GFR_ml_min']),
        'SNGFR_nL_min': float(ctx['SNGFR_nL_min']),
        'MAP_mmHg': float(ctx['mean_arterial_pressure_MAP']),
        'P_bowmans_mmHg': float(ctx['P_bowmans']),
        'steady_state': steady_state,
        'convergence': convergence,
        'parameters': p,
    }
    return result if return_details else result['GFR_ml_min']


In [ ]:
# Baseline confirmation: this should run before any GSA sampling.
baseline_result = run_gfr_model(GSA_INPUT_BASELINE, return_details=True)
baseline_summary = pd.DataFrame({
    'quantity': ['input_dimension', 'GFR_ml_min', 'SNGFR_nL_min', 'MAP_mmHg', 'P_bowmans_mmHg'],
    'value': [
        GSA_INPUT_DIMENSION,
        baseline_result['GFR_ml_min'],
        baseline_result['SNGFR_nL_min'],
        baseline_result['MAP_mmHg'],
        baseline_result['P_bowmans_mmHg'],
    ],
})
baseline_summary.to_csv(OUT_DIR / 'gfr_gsa_86d_baseline_confirmation.csv', index=False)
display(baseline_summary)
print('Saved:', OUT_DIR / 'gfr_gsa_86d_baseline_confirmation.csv')


In [ ]:
# Example: vector interface for OAT/GSA drivers.
x0 = GSA_INPUT_VECTOR_BASELINE
gfr0 = run_gfr_model(x0)
print(f'Baseline GFR from vector input: {gfr0:.6g} mL/min')
